<a href="https://colab.research.google.com/github/QSUM-UNB/EvaporativeCooling/blob/main/PSD_heat_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import scipy.constants as constants
from scipy.constants import pi, speed_of_light as cLight, h, atomic_mass, Boltzmann, epsilon_0, hbar
a_0 = constants.physical_constants['Bohr radius'][0] #Bohr radius [m]
a = 98*a_0 #s-wave scattering length [m] from https://arxiv.org/pdf/1204.1591
m = 86.909180520 * atomic_mass #Rb-87 atomic mass [kg]
taulife = 26.2348E-9 #excited-state lifetime [s]
naturallinewidth = 1/(2*pi*taulife) #natural linewidth [Hz]
wavelength = 1064e-9 #laser wavelength [m]
Delta = cLight/wavelength - cLight/780e-9   #detuning ν_L - ν_0 [Hz]
Delta_omega = 2*np.pi*Delta                 #angular detuning Δω [rad/s]
Gamma_omega = 2*np.pi*naturallinewidth      #natural linewidth Γ [rad/s]
E_r = (h**2/wavelength**2)/(2*m) #recoil energy [J]
k_Boltz = Boltzmann #Boltzmann constant [J/K]
c = cLight #speed of light [m/s]
alpha_ground_si = 7.94e-6*h #ground-state static polarizability α_0 [J/(V/m)^2]
omega_res_THz = 2*pi*377.1074635 #D1 line angular frequency ω_0 [rad/s * 10^12]
omega_res_Hz = omega_res_THz*1e12 #D1 line angular frequency ω_0 [rad/s]
omega_laser = 2*pi*c/wavelength #laser angular frequency ω_L [rad/s]
alpha_detuned = (omega_res_Hz**2 * alpha_ground_si)/(omega_res_Hz**2 - omega_laser**2) #dynamic polarizability α(ω_L) [J/(V/m)^2]
alpha_natural = alpha_detuned/(c*epsilon_0)

In [10]:
def thermal_deBroglie(T, mass=m):
    return np.sqrt(2 * pi * hbar**2 / (mass * Boltzmann * T))

def eta_ev(T, trap_depth):
    return trap_depth / (Boltzmann * T)

def depth(P, waist = 5000e-6, fu = 1.0):
    return 2*((2 * alpha_natural / pi) * (P / waist**2))*fu #extra 2 for 2 beams

def rayleigh(waist):
    return pi * waist**2 / wavelength

def osq_prop(power, waist=5000e-6, fU=1.0):
        trap_depth_val = depth(power, waist, fU)
        zR    = rayleigh(waist)
        return 2 * trap_depth_val / (m * zR**2)

def osq_mod(power, waist=5000e-6, fw=1.0):
        return (8 * alpha_natural * power * fw) / (pi * m * waist**4)

def osq_vert(power, waist=5000e-6, fU=1.0):
      trap_depth_val = depth(power, waist, fU)
      return 4 * trap_depth_val / (m * waist**2)

def ombar(P, waist=5000e-6, fu=1.0):
  osqprop = osq_prop(P, waist, fu)
  osqmod  = osq_mod(P, waist, fu)
  osqvert = osq_vert(P, waist, fu)
  omx = np.sqrt(osqmod + osqprop)
  omy = np.sqrt(osqvert + osqvert)
  omz = np.sqrt(osqmod + osqprop)
  ombar = np.cbrt(omx*omy*omz)
  return(ombar)


def classical_PSD(N, T, P, waist, mass=m):
    trap_depth = depth(P, waist)
    geometric = ombar(P, waist)
    traposcfreq = geometric/(2*np.pi)
    insideterm = (h*traposcfreq)/(k_Boltz*T)
    psd = N*np.power(insideterm, 3)
    return float(psd)

In [11]:
import matplotlib.pyplot as plt
import numpy as np

T_vals = np.arange(0.1e-6, 2e-5 + 0.1e-6, 0.1e-6)
T_labels = {T: f'{T * 1e6:.1f} µK' for T in T_vals}

In [12]:
N_vals = np.logspace(4, 8, 100)
P_vals = np.linspace(0.05, 5.0, 100)

fifty_micro_psd_grids = {}

hundred_micro_psd_grids = {}

one_milli_psd_grids = {}

five_milli_psd_grids = {}

for T_val in T_vals:
    psd_grid_T = np.zeros((len(P_vals), len(N_vals)))
    for i, P_val in enumerate(P_vals):
        for j, N_val in enumerate(N_vals):
            psd_grid_T[i, j] = classical_PSD(N_val, T_val, P_val,50e-6)
    fifty_micro_psd_grids[T_val] = psd_grid_T
for T_val in T_vals:
    psd_grid_T = np.zeros((len(P_vals), len(N_vals)))
    for i, P_val in enumerate(P_vals):
        for j, N_val in enumerate(N_vals):
            psd_grid_T[i, j] = classical_PSD(N_val, T_val, P_val,100e-6)
    hundred_micro_psd_grids[T_val] = psd_grid_T
for T_val in T_vals:
    psd_grid_T = np.zeros((len(P_vals), len(N_vals)))
    for i, P_val in enumerate(P_vals):
        for j, N_val in enumerate(N_vals):
            psd_grid_T[i, j] = classical_PSD(N_val, T_val, P_val,1e-3)
    one_milli_psd_grids[T_val] = psd_grid_T
for T_val in T_vals:
    psd_grid_T = np.zeros((len(P_vals), len(N_vals)))
    for i, P_val in enumerate(P_vals):
        for j, N_val in enumerate(N_vals):
            psd_grid_T[i, j] = classical_PSD(N_val, T_val, P_val,5e-3)
    five_milli_psd_grids[T_val] = psd_grid_T


print("PSD grids calculated for all temperatures.")

PSD grids calculated for all temperatures.


In [13]:
bec_threshold = 2.6
print(f"BEC threshold defined as: {bec_threshold}")

BEC threshold defined as: 2.6


In [14]:
for T_val, psd_grid_T in fifty_micro_psd_grids.items():
    plt.figure(figsize=(10, 8))

    im = plt.pcolormesh(N_vals, 2*P_vals, psd_grid_T, shading='auto', cmap='viridis', vmin=0, vmax=10)
    plt.colorbar(im, label='Phase Space Density (PSD)')

    plt.xscale('log')

    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[bec_threshold], colors='red', linestyles='dashed', linewidths=2)
    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[1], colors='orange', linestyles='dashed', linewidths=2)
    plt.axvline(1e6, color='white', linestyle='--', linewidth=2, label='N = 1e6')
    plt.xlabel('Atom Number (N)')
    plt.ylabel('Optical Trap Power (P)')
    plt.title(f'PSD Heatmap for T = {T_labels[T_val]}, W = 50μm')
    plt.text(N_vals.min(), P_vals.min(), f'BEC threshold: {bec_threshold}', color='red', verticalalignment='bottom', horizontalalignment='left', bbox=dict(facecolor='white', alpha=0.7))

    filename_png = f'50_micro_psd_heatmap_T_{T_val * 1e6:.1f}uK.png'
    plt.savefig(filename_png)
    plt.close()
for T_val, psd_grid_T in hundred_micro_psd_grids.items():
    plt.figure(figsize=(10, 8))

    im = plt.pcolormesh(N_vals, 2*P_vals, psd_grid_T, shading='auto', cmap='viridis', vmin=0, vmax=10)
    plt.colorbar(im, label='Phase Space Density (PSD)')

    plt.xscale('log')

    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[bec_threshold], colors='red', linestyles='dashed', linewidths=2)
    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[1], colors='orange', linestyles='dashed', linewidths=2)
    plt.axvline(1e6, color='white', linestyle='--', linewidth=2, label='N = 1e6')
    plt.xlabel('Atom Number (N)')
    plt.ylabel('Optical Trap Power (P)')
    plt.title(f'PSD Heatmap for T = {T_labels[T_val]},W = 100μm')
    plt.text(N_vals.min(), P_vals.min(), f'BEC threshold: {bec_threshold}', color='red', verticalalignment='bottom', horizontalalignment='left', bbox=dict(facecolor='white', alpha=0.7))

    filename_png = f'100_micro_psd_heatmap_T_{T_val * 1e6:.1f}uK.png'
    plt.savefig(filename_png)
    plt.close()
for T_val, psd_grid_T in one_milli_psd_grids.items():
    plt.figure(figsize=(10, 8))

    im = plt.pcolormesh(N_vals, 2*P_vals, psd_grid_T, shading='auto', cmap='viridis', vmin=0, vmax=10)
    plt.colorbar(im, label='Phase Space Density (PSD)')

    plt.xscale('log')

    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[bec_threshold], colors='red', linestyles='dashed', linewidths=2)
    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[1], colors='orange', linestyles='dashed', linewidths=2)
    plt.axvline(1e6, color='white', linestyle='--', linewidth=2, label='N = 1e6')
    plt.xlabel('Atom Number (N)')
    plt.ylabel('Optical Trap Power (P)')
    plt.title(f'PSD Heatmap for T = {T_labels[T_val]}, W = 1000μm')
    plt.text(N_vals.min(), P_vals.min(), f'BEC threshold: {bec_threshold}', color='red', verticalalignment='bottom', horizontalalignment='left', bbox=dict(facecolor='white', alpha=0.7))

    filename_png = f'1mm_psd_heatmap_T_{T_val * 1e6:.1f}uK.png'
    plt.savefig(filename_png)
    plt.close()
for T_val, psd_grid_T in five_milli_psd_grids.items():
    plt.figure(figsize=(10, 8))

    im = plt.pcolormesh(N_vals, 2*P_vals, psd_grid_T, shading='auto', cmap='viridis', vmin=0, vmax=10)
    plt.colorbar(im, label='Phase Space Density (PSD)')

    plt.xscale('log')

    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[bec_threshold], colors='red', linestyles='dashed', linewidths=2)
    plt.contour(N_vals, 2*P_vals, psd_grid_T, levels=[1], colors='orange', linestyles='dashed', linewidths=2)
    plt.axvline(1e6, color='white', linestyle='--', linewidth=2, label='N = 1e6')
    plt.xlabel('Atom Number (N)')
    plt.ylabel('Optical Trap Power (P)')
    plt.title(f'PSD Heatmap for T = {T_labels[T_val]}, W = 5000 μm')
    plt.text(N_vals.min(), P_vals.min(), f'BEC threshold: {bec_threshold}', color='red', verticalalignment='bottom', horizontalalignment='left', bbox=dict(facecolor='white', alpha=0.7))

    filename_png = f'5mm_psd_heatmap_T_{T_val * 1e6:.1f}uK.png'
    plt.savefig(filename_png)
    plt.close()